# Building a Production Whisper Transcription Pipeline

The OpenAI Whisper API (`audio.transcriptions.create`) is straightforward for short clips, but production workloads involve longer audio, unreliable input, and the need to track progress. This notebook shows how to build a robust pipeline that handles:

1. **Chunking** — splitting audio that exceeds the 25 MB API limit
2. **Progress tracking** — knowing how far along transcription is
3. **Error handling** — retrying failed chunks without restarting everything
4. **Post-processing** — using an agent to clean and structure the transcript
5. **Cost estimation** — calculating the cost per minute of audio

## Setup

In [ ]:
%pip install openai openai-agents pydub --quiet

In [ ]:
import asyncio
import io
import os
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable, Optional

import openai
from agents import Agent, Runner, function_tool

client = openai.AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Whisper API hard limit is 25 MB. We use 20 MB to leave a safe margin.
WHISPER_MAX_BYTES = 20 * 1024 * 1024
# Whisper costs $0.006 per minute of audio (as of 2025)
WHISPER_COST_PER_MINUTE = 0.006

## Part 1 — Simple Transcription with Progress Tracking

For short audio files that fit within the API limit, the simplest approach is a single API call. We add a progress callback pattern so callers always know what's happening.

In [ ]:
async def transcribe_file(
    audio_path: str | Path,
    model: str = "whisper-1",
    language: Optional[str] = None,
    on_progress: Optional[Callable[[str], None]] = None,
) -> str:
    """Transcribe a single audio file using the Whisper API.

    Args:
        audio_path: Path to the audio file (mp3, wav, m4a, etc.).
        model: Whisper model to use. Currently only 'whisper-1' is available.
        language: Optional BCP-47 language code (e.g. 'en', 'fr'). Auto-detected if None.
        on_progress: Optional callback called with status messages.

    Returns:
        The transcribed text.
    """
    path = Path(audio_path)
    file_size = path.stat().st_size

    if on_progress:
        on_progress(f"Starting transcription of {path.name} ({file_size / 1024 / 1024:.1f} MB)")

    if file_size > WHISPER_MAX_BYTES:
        raise ValueError(
            f"File is {file_size / 1024 / 1024:.1f} MB, exceeding the {WHISPER_MAX_BYTES / 1024 / 1024:.0f} MB limit. "
            "Use transcribe_long_audio() instead."
        )

    with open(path, "rb") as f:
        kwargs = {"model": model, "file": f}
        if language:
            kwargs["language"] = language

        if on_progress:
            on_progress("Sending to Whisper API...")

        response = await client.audio.transcriptions.create(**kwargs)

    if on_progress:
        on_progress("Transcription complete.")

    return response.text


# Example usage (replace with a real audio file path)
# transcript = await transcribe_file(
#     "my_audio.mp3",
#     on_progress=lambda msg: print(f"[progress] {msg}"),
# )
print("transcribe_file() defined — ready to use with any audio file under 20 MB.")

## Part 2 — Chunked Transcription for Long Audio

Audio files longer than ~30 minutes often exceed the 25 MB limit. The solution is to split the file into chunks, transcribe each one, and join the results.

In [ ]:
@dataclass
class TranscriptionChunk:
    index: int
    start_ms: int
    end_ms: int
    text: str = ""
    failed: bool = False
    error: str = ""


async def transcribe_long_audio(
    audio_path: str | Path,
    chunk_duration_ms: int = 10 * 60 * 1000,  # 10 minutes per chunk
    model: str = "whisper-1",
    language: Optional[str] = None,
    on_progress: Optional[Callable[[int, int, str], None]] = None,
    max_retries: int = 2,
) -> str:
    """Transcribe a long audio file by splitting it into chunks.

    Args:
        audio_path: Path to the audio file.
        chunk_duration_ms: Duration of each chunk in milliseconds.
        model: Whisper model to use.
        language: Optional language code.
        on_progress: Callback called as (completed_chunks, total_chunks, status).
        max_retries: How many times to retry a failed chunk before marking it as failed.

    Returns:
        Full transcript joined from all chunks.
    """
    try:
        from pydub import AudioSegment
    except ImportError:
        raise ImportError("Install pydub to transcribe long audio: pip install pydub")

    path = Path(audio_path)
    audio = AudioSegment.from_file(str(path))
    duration_ms = len(audio)

    # Build chunk list
    chunks: list[TranscriptionChunk] = []
    start = 0
    idx = 0
    while start < duration_ms:
        end = min(start + chunk_duration_ms, duration_ms)
        chunks.append(TranscriptionChunk(index=idx, start_ms=start, end_ms=end))
        start = end
        idx += 1

    total = len(chunks)
    if on_progress:
        on_progress(0, total, f"Split into {total} chunks ({chunk_duration_ms // 60000} min each)")

    for chunk in chunks:
        segment = audio[chunk.start_ms : chunk.end_ms]

        # Export chunk to an in-memory buffer so we don't write temp files
        buf = io.BytesIO()
        segment.export(buf, format="mp3")
        buf.seek(0)
        buf.name = f"chunk_{chunk.index}.mp3"

        for attempt in range(max_retries + 1):
            try:
                kwargs = {"model": model, "file": buf}
                if language:
                    kwargs["language"] = language
                response = await client.audio.transcriptions.create(**kwargs)
                chunk.text = response.text
                break
            except Exception as exc:
                if attempt == max_retries:
                    chunk.failed = True
                    chunk.error = str(exc)
                else:
                    buf.seek(0)
                    await asyncio.sleep(2 ** attempt)  # exponential backoff

        if on_progress:
            status = "failed" if chunk.failed else "done"
            on_progress(
                chunk.index + 1, total,
                f"Chunk {chunk.index + 1}/{total} {status}"
                + (f": {chunk.error}" if chunk.failed else "")
            )

    # Report any failed chunks
    failed = [c for c in chunks if c.failed]
    if failed:
        print(f"Warning: {len(failed)} chunk(s) failed and will appear as [transcription failed]:")
        for c in failed:
            c.text = f"[transcription failed for segment {c.index + 1}: {c.error}]"

    return " ".join(c.text for c in chunks)


print("transcribe_long_audio() defined — handles chunking, retries, and progress tracking.")

## Part 3 — Cost Estimation

Whisper is priced per minute of audio. Estimate costs before and after transcription.

In [ ]:
def estimate_transcription_cost(duration_seconds: float) -> float:
    """Estimate Whisper transcription cost in USD.

    Args:
        duration_seconds: Audio duration in seconds.

    Returns:
        Estimated cost in USD.
    """
    duration_minutes = duration_seconds / 60
    return duration_minutes * WHISPER_COST_PER_MINUTE


# Example: 1 hour podcast
duration = 60 * 60  # 3600 seconds
cost = estimate_transcription_cost(duration)
print(f"Estimated cost for a {duration // 60}-minute audio file: ${cost:.4f}")

# 10-minute clip
cost_10min = estimate_transcription_cost(600)
print(f"Estimated cost for a 10-minute audio file: ${cost_10min:.4f}")

## Part 4 — Post-Processing with an Agent

Raw Whisper transcripts often need cleanup: removing filler words, fixing formatting, or extracting structured information. An agent is perfect for this.

In [ ]:
from pydantic import BaseModel


class MeetingNotes(BaseModel):
    summary: str
    action_items: list[str]
    key_decisions: list[str]
    participants_mentioned: list[str]


meeting_notes_agent = Agent(
    name="Meeting Notes Agent",
    instructions=(
        "You are an expert at extracting structured meeting notes from raw transcripts. "
        "Given a raw meeting transcript, extract: a concise summary, action items, "
        "key decisions made, and any names mentioned."
    ),
    output_type=MeetingNotes,
    model="gpt-4o-mini",
)


async def process_transcript(raw_transcript: str) -> MeetingNotes:
    """Extract structured meeting notes from a raw Whisper transcript.

    Args:
        raw_transcript: The raw text output from Whisper.

    Returns:
        Structured MeetingNotes object.
    """
    result = await Runner.run(meeting_notes_agent, raw_transcript)
    return result.final_output


# Demo with a synthetic transcript
sample_transcript = """
Uh, okay so I think we should, um, launch the new feature by end of Q3.
Sarah mentioned that the design is already done. John said he needs two more weeks
for backend work. We decided to skip the A/B test for now and go straight to full rollout.
Action items: John to finish the API by July 15, Sarah to hand off designs to the team,
and Ahmed to set up monitoring dashboards before launch.
"""

notes = await process_transcript(sample_transcript)
print("Structured meeting notes:")
print(f"  Summary: {notes.summary}")
print(f"  Action items:")
for item in notes.action_items:
    print(f"    - {item}")
print(f"  Key decisions:")
for decision in notes.key_decisions:
    print(f"    - {decision}")
print(f"  Participants: {', '.join(notes.participants_mentioned)}")

## Part 5 — Putting It All Together

A complete pipeline function that transcribes audio and extracts structured notes in one call.

In [ ]:
async def transcribe_and_extract(
    audio_path: str | Path,
    language: Optional[str] = None,
    verbose: bool = True,
) -> tuple[str, MeetingNotes]:
    """Full pipeline: transcribe audio then extract structured meeting notes.

    Args:
        audio_path: Path to audio file.
        language: Optional language code for Whisper.
        verbose: Whether to print progress updates.

    Returns:
        Tuple of (raw_transcript, structured_notes).
    """
    path = Path(audio_path)
    file_size = path.stat().st_size

    def log(msg: str) -> None:
        if verbose:
            print(f"[pipeline] {msg}")

    log(f"Processing: {path.name}")
    start = time.time()

    if file_size <= WHISPER_MAX_BYTES:
        raw = await transcribe_file(
            path,
            language=language,
            on_progress=log,
        )
    else:
        raw = await transcribe_long_audio(
            path,
            language=language,
            on_progress=lambda done, total, msg: log(msg),
        )

    log(f"Transcription done in {time.time() - start:.1f}s. Extracting notes...")
    notes = await process_transcript(raw)
    log("Pipeline complete.")

    return raw, notes


print("transcribe_and_extract() defined.")
print("Usage: raw, notes = await transcribe_and_extract('meeting.mp3')")

## Summary

| Component | What it does |
|---|---|
| `transcribe_file()` | Single file transcription with progress callback |
| `transcribe_long_audio()` | Splits large files into chunks, retries failures, tracks progress |
| `estimate_transcription_cost()` | Estimates USD cost before making API calls |
| `process_transcript()` | Agent-powered structured extraction from raw transcript |
| `transcribe_and_extract()` | Full pipeline combining all of the above |

### Key takeaways
- Always check file size before calling the API — files over ~25 MB will be rejected
- Use in-memory buffers (`io.BytesIO`) for chunk export to avoid writing temp files to disk
- Exponential backoff on retries (`2 ** attempt` seconds) handles transient API errors
- An agent with a Pydantic `output_type` is the cleanest way to structure raw transcripts